Будем выводить на экран дневной график акций Сбербанка за последний год

In [1]:
from datetime import date, timedelta  # Диапазон дат за последний год

dataname = 'TQBR.SBER'  # Тикер
year_ago = date.today() - timedelta(days=365)  # Год назад

Получим брокера и историю тикера из библиотеки FinLabPy

In [2]:
from FinLabPy.Config import brokers, default_broker  # Все брокеры и брокер по умолчанию
from FinLabPy.BackTrader import Store, PlotLC  # Хранилище BackTrader, построение графика Lightweight Charts

store = Store(broker=default_broker)  # Хранилище брокера по умолчанию
# store = Store(broker=brokers['<Ключ словаря brokers из Config.py>'])  # Хранилище выбранного брокера
broker = store.getbroker()  # Брокер
data = store.getdata(dataname=dataname, fromdate=year_ago)  # История тикера

Индикаторы на графике будем показывать через торговую систему

In [3]:
import backtrader as bt  # Основная библиотека BackTrader
from backtrader.indicators import MovingAverageSimple, RelativeStrengthIndex, Momentum  # Классические индикаторы SMA, RSI, Momentum

class PlotIndicators(bt.Strategy):
    def __init__(self):
        self.sma = MovingAverageSimple(self.data.close, period=100)  # SMA
        self.rsi = RelativeStrengthIndex(self.data.close, period=14)  # RSI
        self.momentum = Momentum(self.data.close, period=1)  # Momentum

Настроим и запустим "движок" BackTrader

In [4]:
# cerebro = bt.Cerebro()  # Инициируем "движок" BackTrader
cerebro = bt.Cerebro(stdstats=False)  # Инициируем "движок" BackTrader. Стандартная статистика сделок и кривой доходности не нужна
cerebro.setbroker(broker)  # Устанавливаем брокера
cerebro.adddata(data)  # Привязываем исторические данные
cerebro.addstrategy(PlotIndicators)  # Привязываем индикаторы через торговую систему
cerebro.run()  # Запуск "движка" BackTrader

Нарисуем стандартный график BackTrader

In [5]:
%matplotlib inline

import matplotlib as mpl
mpl.rcParams['figure.facecolor'] = 'grey'  # По умолчанию в стандартном графике получаем черную линию цен на черном фоне
mpl.rcParams['axes.facecolor'] = 'grey'  # Поэтому, поменяем фон на серый, чтобы было нормально видно

figs = cerebro.plot(volume=False)  # Все стандартные графики BackTrader после запуска
fig = figs[0][0] # Первый график

<IPython.core.display.Javascript object>

Нарисуем график Lightweight Charts

In [6]:
run_strat = cerebro.runstrats[0][0]  # ТС с результатами запуска
setattr(run_strat.sma.plotinfo, 'lines', {'sma': {'style': 'solid', 'color': 'blue'}})  # style='solid'/'dotted'/'dashed'/'large_dashed'/'sparse_dotted'
setattr(run_strat.rsi.plotinfo, 'lines', {'rsi': {'pane_id': 1, 'color': 'green'}})
setattr(run_strat.momentum.plotinfo, 'lines', {'momentum': {'pane_id': 2, 'color': 'red'}})

cerebro.plot(plotter=PlotLC.Plot(volume=False))  # Рисуем график Lightweight Charts

[None]